In [1]:
import os
os.environ['PYTHONHASHSEED'] = '0'
import warnings
warnings.filterwarnings('ignore')

import json as _json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from astropy.timeseries import LombScargle
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, f1_score, fbeta_score, accuracy_score,
)

seed = 42
np.random.seed(seed); torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CHANNELS       = ['rv']                  # ['rv','RHKp','Halpha'] for ablation
N_FREQ         = 500                      # matches rf_multiseed V4 LS grid
LS_METHOD      = 'fast'                   # astropy backend (matches V4)

N_REPS   = 5
N_FOLDS  = 5
N_EPOCHS = 60
BATCH    = 32
LR       = 3e-4
WD       = 1e-4

OBS_PKL = 'data/observations.pkl'

observations = pd.read_pickle(OBS_PKL)
print(f'Total observations: {len(observations)}')
print(f'Stars: {observations["star_name"].nunique()}')
print(f'Columns: {list(observations.columns)}')
print(f'Using device: {device}')

Total observations: 220318
Stars: 2026
Columns: ['star_name', 'bjd', 'rv', 'rv_err', 'exposure_time', 'RHKp', 'Halpha', 'has_exoplanets', 'rv_centered']
Using device: cuda


In [2]:
grouped  = observations.groupby('star_name', sort=True)
stars    = list(grouped.groups.keys())
labels   = np.array([int(grouped.get_group(s)['has_exoplanets'].iloc[0]) for s in stars],
                    dtype=int)
n_stars  = len(stars)
print(f'n_stars = {n_stars}, positives = {int(labels.sum())}, negatives = {int((labels==0).sum())}')
print(f'Class ratio 1:{n_stars/max(labels.sum(),1):.1f}')

n_stars = 2026, positives = 430, negatives = 1596
Class ratio 1:4.7


In [3]:
obs_per_star = observations.groupby('star_name', sort=True).agg(
    bjd_min=('bjd','min'), bjd_max=('bjd','max'), n=('bjd','count'))
baselines = obs_per_star['bjd_max'] - obs_per_star['bjd_min']

max_baseline = max(float(baselines.max()), 2.0)
f_low  = 1.0 / max_baseline
f_high = 1.0   # period >= 1 day, exactly V4's bound
freq_grid = np.linspace(f_low, f_high, N_FREQ)
print(f'Grid: {N_FREQ} freqs in [{f_low:.4f}, {f_high:.4f}] cyc/day')
print(f'  Periods: [{1/f_high:.2f}, {1/f_low:.2f}] days')
print(f'  Max baseline across stars: {max_baseline:.0f} days ({max_baseline/365.25:.1f} yrs)')

CHAN_COL = {'rv': 'rv_centered', 'RHKp': 'RHKp', 'Halpha': 'Halpha'}
used_cols = [CHAN_COL[c] for c in CHANNELS]
print(f'Channels ({len(CHANNELS)}): {CHANNELS} -> cols {used_cols}')

star_groups = {s: grouped.get_group(s).sort_values('bjd') for s in stars}

def compute_pgram(star):
    g = star_groups[star]
    bjd  = g['bjd'].values.astype(float)
    spectra = []
    for chan in CHANNELS:
        vals = g[CHAN_COL[chan]].values.astype(float)
        if len(bjd) < 2 or not np.all(np.isfinite(vals)) or np.std(vals) < 1e-8:
            spectra.append(np.zeros(N_FREQ, dtype=np.float32))
            continue
        dy = None
        if chan == 'rv':
            e = g['rv_err'].values.astype(float)
            if np.all(np.isfinite(e)) and np.all(e > 0):
                dy = e
        ls = LombScargle(bjd, vals, dy=dy, fit_mean=True, center_data=True)
        power = ls.power(freq_grid, method=LS_METHOD, normalization='standard')
        power = np.nan_to_num(power.astype(np.float32), nan=0.0, posinf=1.0, neginf=0.0)
        power = np.clip(power, 0.0, 1.0)
        spectra.append(power.astype(np.float32))
    return np.stack(spectra, axis=0)  # (C, N_FREQ)

print(f'Computing {n_stars} x {len(CHANNELS)}-channel periodograms on {N_FREQ} bins...')
star_to_pgram = {s: compute_pgram(s) for s in stars}
print('Done.')

pg_array = np.stack([star_to_pgram[s] for s in stars], axis=0).astype(np.float32)
print(f'pg_array shape: {pg_array.shape}')

pg_array = np.log1p(pg_array)  # log(1 + power)

flat = pg_array.reshape(pg_array.shape[0], -1)            # (n_stars, C*N_FREQ)
p_lo  = np.percentile(flat, 1.0)
p_hi  = np.percentile(flat, 99.0)
scale = max(p_hi - p_lo, 1e-6)
shift = (p_lo + p_hi) / 2.0
pg_array = (pg_array - shift) / scale
pg_array = np.clip(pg_array, -5.0, 5.0).astype(np.float32)
pg_array = np.nan_to_num(pg_array, nan=0.0, posinf=5.0, neginf=-5.0)
print(f'log1p + percentile standardized. mean={pg_array.mean():.4f}, std={pg_array.std():.4f}')
print(f'  percentiles: p1={p_lo:.4f}, p99={p_hi:.4f}, scale={scale:.4f}')
assert np.isfinite(pg_array).all(), 'pg_array still has NaN/Inf after sanitization'
assert pg_array.std() > 1e-3, f'STANDARDIZATION COLLAPSED ARRAY: std={pg_array.std():.6f}'
print(f'  final range: [{pg_array.min():.3f}, {pg_array.max():.3f}]')

Grid: 500 freqs in [0.0001, 1.0000] cyc/day
  Periods: [1.00, 8643.19] days
  Max baseline across stars: 8643 days (23.7 yrs)
Channels (1): ['rv'] -> cols ['rv_centered']
Computing 2026 x 1-channel periodograms on 500 bins...
Done.
pg_array shape: (2026, 1, 500)
log1p + percentile standardized. mean=-0.3533, std=0.1936
  percentiles: p1=0.0002, p99=0.5427, scale=0.5424
  final range: [-0.500, 0.777]


In [4]:
class PgramDataset(Dataset):
    def __init__(self, idx_list, y):
        self.idx = list(idx_list)
        self.y  = y.astype(np.float32)
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        return torch.from_numpy(pg_array[self.idx[i]]), torch.tensor(self.y[self.idx[i]]).float()

class ConvBlock(nn.Module):
    """Conv1d + GroupNorm(1, co) + GELU + Dropout + 2x AvgPool."""
    def __init__(self, ci, co, k, drop=0.3):
        super().__init__()
        self.conv = nn.Conv1d(ci, co, k, padding=k//2)
        self.norm = nn.GroupNorm(1, co)   # LayerNorm over channels
        self.act  = nn.GELU()
        self.drop = nn.Dropout(drop)
        self.pool = nn.AvgPool1d(2)
    def forward(self, x):
        return self.pool(self.drop(self.act(self.norm(self.conv(x)))))

class PgramCNN(nn.Module):
    def __init__(self, in_ch=1, hidden=32, drop=0.3):
        super().__init__()
        self.b1 = ConvBlock(in_ch,      hidden,    7, drop)
        self.b2 = ConvBlock(hidden,     hidden*2,  5, drop)
        self.b3 = ConvBlock(hidden*2,   hidden*4,  5, drop)
        self.b4 = ConvBlock(hidden*4,   hidden*4,  3, drop)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.gmp = nn.AdaptiveMaxPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(2 * hidden*4, 32),
            nn.GELU(), nn.Dropout(drop),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        h = self.b4(self.b3(self.b2(self.b1(x))))   # (B, C, L)
        a = self.gap(h).squeeze(-1)
        m = self.gmp(h).squeeze(-1)
        return self.head(torch.cat([a, m], dim=1)).squeeze(-1)  # (B,)

model = PgramCNN(in_ch=len(CHANNELS)).to(device)
n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'PgramCNN parameters: {n_p:,}')
x0 = torch.randn(BATCH, len(CHANNELS), N_FREQ).to(device)
y0 = model(x0)
print(f'smoke forward: in={tuple(x0.shape)} out={tuple(y0.shape)}')

PgramCNN parameters: 109,889
smoke forward: in=(32, 1, 500) out=(32,)


In [5]:
def train_one_fold(train_idx, test_idx, rep_seed, verbose=False):
    """Train a fresh CNN on train_idx; return probabilities on test_idx (held-out)."""
    torch.manual_seed(rep_seed); torch.cuda.manual_seed_all(rep_seed)
    np.random.seed(rep_seed)

    model = PgramCNN(in_ch=len(CHANNELS)).to(device)
    pos = max(int((labels[train_idx]==1).sum()), 1)
    neg = max(int((labels[train_idx]==0).sum()), 1)
    pos_w = torch.tensor([neg/pos], device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, epochs=N_EPOCHS, steps_per_epoch=1, pct_start=0.1)

    ds_tr = PgramDataset(train_idx, labels)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH, shuffle=True, drop_last=False)
    model.train()
    for ep in range(N_EPOCHS):
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward(); opt.step()
        sched.step()
        if verbose and (ep+1) % 10 == 0:
            print(f'    ep {ep+1:3d}: loss={loss.item():.4f}')

    model.eval()
    with torch.no_grad():
        x_te = torch.from_numpy(pg_array[test_idx]).to(device)
        logits = model(x_te).cpu().numpy()
    logits = np.nan_to_num(logits, nan=0.0, posinf=35.0, neginf=-35.0)
    probs = 1.0 / (1.0 + np.exp(-logits))
    probs = np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0).astype(np.float32)
    return probs


In [6]:
all_oof_probs = np.zeros((n_stars, N_REPS))
all_oof_preds = np.zeros((n_stars, N_REPS), dtype=int)
rep_metrics   = {'rep': [], 'pr_auc': [], 'roc_auc': [],
                 'f1': [], 'f05': [], 'precision': [], 'recall': []}

import time
for rep in range(N_REPS):
    oof_preds = np.zeros(n_stars, dtype=int)
    oof_probs = np.zeros(n_stars)
    rep_seed  = 42 + rep
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=rep_seed)
    fold_times = []
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(np.zeros(n_stars), labels)):
        t0 = time.time()

        inner_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=rep_seed)
        inner_probs_fold = np.zeros(len(train_idx), dtype=np.float32)
        for i_train, i_val in inner_skf.split(np.zeros(len(train_idx)), labels[train_idx]):
            inner_probs_fold[i_val] = train_one_fold(train_idx[i_train], train_idx[i_val], rep_seed)

        vp_in, vr_in, vt_in = precision_recall_curve(labels[train_idx], inner_probs_fold)
        vf1_in = 2 * vp_in * vr_in / (vp_in + vr_in + 1e-8)
        fold_thr = float(vt_in[int(np.argmax(vf1_in))]) if len(vt_in) > 0 else 0.5

        probs = train_one_fold(train_idx, test_idx, rep_seed, verbose=False)
        oof_probs[test_idx] = probs
        oof_preds[test_idx] = (probs >= fold_thr).astype(int)
        fold_times.append(time.time() - t0)
        if (fold_idx+1) % 2 == 0:
            print(f'  rep {rep} fold {fold_idx+1:2d}/{N_FOLDS} done ({np.mean(fold_times):.0f}s/fold avg)')
        del probs
    all_oof_probs[:, rep] = oof_probs
    all_oof_preds[:, rep] = oof_preds

    roc = roc_auc_score(labels, oof_probs)
    pr  = average_precision_score(labels, oof_probs)
    cm = confusion_matrix(labels, oof_preds)
    tn, fp, fn, tp = cm.ravel()
    prc = tp / (tp+fp) if (tp+fp) > 0 else 0.0
    rec = tp / (tp+fn) if (tp+fn) > 0 else 0.0
    f1  = f1_score(labels, oof_preds, zero_division=0)
    f05 = fbeta_score(labels, oof_preds, beta=0.5, zero_division=0)
    rep_metrics['rep'].append(rep); rep_metrics['pr_auc'].append(pr)
    rep_metrics['roc_auc'].append(roc); rep_metrics['f1'].append(f1)
    rep_metrics['f05'].append(f05)
    rep_metrics['precision'].append(prc); rep_metrics['recall'].append(rec)
    print(f'  rep {rep} (seed {rep_seed}): pr_auc={pr:.4f} roc_auc={roc:.4f} f1={f1:.4f} f0.5={f05:.4f} p={prc:.3f} r={rec:.3f}')

  rep 0 fold  2/5 done (48s/fold avg)
  rep 0 fold  4/5 done (46s/fold avg)
  rep 0 (seed 42): pr_auc=0.3290 roc_auc=0.6818 f1=0.4482 f0.5=0.3723 p=0.334 r=0.679
  rep 1 fold  2/5 done (44s/fold avg)
  rep 1 fold  4/5 done (44s/fold avg)
  rep 1 (seed 43): pr_auc=0.3373 roc_auc=0.6892 f1=0.4505 f0.5=0.3752 p=0.338 r=0.677
  rep 2 fold  2/5 done (43s/fold avg)
  rep 2 fold  4/5 done (42s/fold avg)
  rep 2 (seed 44): pr_auc=0.3305 roc_auc=0.6864 f1=0.4462 f0.5=0.3726 p=0.336 r=0.665
  rep 3 fold  2/5 done (43s/fold avg)
  rep 3 fold  4/5 done (42s/fold avg)
  rep 3 (seed 45): pr_auc=0.3258 roc_auc=0.6778 f1=0.4341 f0.5=0.3597 p=0.323 r=0.663
  rep 4 fold  2/5 done (43s/fold avg)
  rep 4 fold  4/5 done (43s/fold avg)
  rep 4 (seed 46): pr_auc=0.3333 roc_auc=0.6821 f1=0.4460 f0.5=0.3728 p=0.336 r=0.663


In [7]:
rep_df = pd.DataFrame(rep_metrics)

print("\nper-rep pr_auc:")
for _, row in rep_df.iterrows():
    print(f"  rep {int(row['rep'])}: pr_auc={row['pr_auc']:.4f}")

print(f"\naggregate (n={N_REPS} reps):")
for m in ['pr_auc','roc_auc','f1','f05','precision','recall']:
    v = rep_df[m].values
    print(f"  {m}: {v.mean():.4f} +/- {v.std(ddof=1):.4f} (min={v.min():.4f}, max={v.max():.4f})")

avg_oof = all_oof_probs.mean(axis=1)
combined_pr  = average_precision_score(labels, avg_oof)
combined_roc = roc_auc_score(labels, avg_oof)
combined_preds = (all_oof_preds.mean(axis=1) >= 0.5).astype(int)
combined_f1   = f1_score(labels, combined_preds, zero_division=0)
combined_f05  = fbeta_score(labels, combined_preds, beta=0.5, zero_division=0)
cm = confusion_matrix(labels, combined_preds)
tn, fp, fn, tp = cm.ravel()
combined_p = tp/(tp+fp) if (tp+fp) > 0 else 0.0
combined_r = tp/(tp+fn) if (tp+fn) > 0 else 0.0

print("\ncombined oof (avg across reps):")
print(f"  pr_auc: {combined_pr:.4f}")
print(f"  roc_auc: {combined_roc:.4f}")
print(f"  f1: {combined_f1:.4f}  f0.5: {combined_f05:.4f}")
print(f"  p={combined_p:.3f}  r={combined_r:.3f} (TN={tn} FP={fp} FN={fn} TP={tp})")

def bootstrap_roc_auc(y_true, y_score, n_resamples=200, seed=42):
    """Bootstrap 95% CI for ROC-AUC via the percentile method."""
    from sklearn.metrics import roc_auc_score

    y_true = np.asarray(y_true).ravel()
    y_score = np.asarray(y_score).ravel()
    if len(y_true) != len(y_score):
        raise ValueError(f"length mismatch: y_true={len(y_true)} y_score={len(y_score)}")
    if len(y_true) < 2:
        raise ValueError("need at least 2 samples to bootstrap ROC-AUC")

    point = float(roc_auc_score(y_true, y_score))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs = np.empty(n_resamples, dtype=float)
    for i in range(n_resamples):
        sample_idx = rng.integers(0, n, size=n)
        yt = y_true[sample_idx]
        ys = y_score[sample_idx]
        attempts = 0
        while len(np.unique(yt)) < 2 and attempts < 10:
            sample_idx = rng.integers(0, n, size=n)
            yt = y_true[sample_idx]
            ys = y_score[sample_idx]
            attempts += 1
        if len(np.unique(yt)) < 2:
            aucs[i] = point  # fall back to point estimate if degenerate
            continue
        aucs[i] = roc_auc_score(yt, ys)
    lo = float(np.percentile(aucs, 2.5))
    hi = float(np.percentile(aucs, 97.5))
    return point, lo, hi

def bootstrap_pr_auc(y_true, y_score, n_resamples=200, seed=42):
    """Bootstrap 95% CI for PR-AUC (average precision) via the percentile method."""
    from sklearn.metrics import average_precision_score

    y_true = np.asarray(y_true).ravel()
    y_score = np.asarray(y_score).ravel()
    if len(y_true) != len(y_score):
        raise ValueError(f"length mismatch: y_true={len(y_true)} y_score={len(y_score)}")
    if len(y_true) < 2:
        raise ValueError("need at least 2 samples to bootstrap PR-AUC")

    point = float(average_precision_score(y_true, y_score))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aps = np.empty(n_resamples, dtype=float)
    for i in range(n_resamples):
        sample_idx = rng.integers(0, n, size=n)
        yt = y_true[sample_idx]
        ys = y_score[sample_idx]
        attempts = 0
        while len(np.unique(yt)) < 2 and attempts < 10:
            sample_idx = rng.integers(0, n, size=n)
            yt = y_true[sample_idx]
            ys = y_score[sample_idx]
            attempts += 1
        if len(np.unique(yt)) < 2:
            aps[i] = point  # fall back to point estimate if degenerate
            continue
        aps[i] = average_precision_score(yt, ys)
    lo = float(np.percentile(aps, 2.5))
    hi = float(np.percentile(aps, 97.5))
    return point, lo, hi
pr_point, pr_lo, pr_hi = bootstrap_pr_auc(labels, avg_oof)
roc_point, roc_lo, roc_hi = bootstrap_roc_auc(labels, avg_oof)
print("\nbootstrap 95% ci (200 resamples):")
print(f"  pr_auc: {pr_point:.4f} [{pr_lo:.4f}, {pr_hi:.4f}]")
print(f"  roc_auc: {roc_point:.4f} [{roc_lo:.4f}, {roc_hi:.4f}]")


per-rep pr_auc:
  rep 0: pr_auc=0.3290
  rep 1: pr_auc=0.3373
  rep 2: pr_auc=0.3305
  rep 3: pr_auc=0.3258
  rep 4: pr_auc=0.3333

aggregate (n=5 reps):
  pr_auc: 0.3312 +/- 0.0044 (min=0.3258, max=0.3373)
  roc_auc: 0.6835 +/- 0.0044 (min=0.6778, max=0.6892)
  f1: 0.4450 +/- 0.0063 (min=0.4341, max=0.4505)
  f05: 0.3705 +/- 0.0062 (min=0.3597, max=0.3752)
  precision: 0.3333 +/- 0.0060 (min=0.3228, max=0.3376)
  recall: 0.6693 +/- 0.0080 (min=0.6628, max=0.6791)

combined oof (avg across reps):
  pr_auc: 0.3230
  roc_auc: 0.6840
  f1: 0.4499  f0.5: 0.3742
  p=0.336  r=0.679 (TN=1020 FP=576 FN=138 TP=292)

bootstrap 95% ci (200 resamples):
  pr_auc: 0.3230 [0.2920, 0.3689]
  roc_auc: 0.6840 [0.6595, 0.7140]
